## 1. Setup


In [ ]:
import copy
import itertools
import platform
from collections import OrderedDict
from datetime import datetime
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.readwrite import BIFReader, BIFWriter, XMLBIFReader, XMLBIFWriter

## 2. Configuration


In [ ]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "/data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

MODEL_PATH = DATA_DIR / "breast_treat_qol_likert.bif"
EVIDENCE_PATH = DATA_DIR / "cpt_likert_csv.csv"

LEARNED_NETWORK_DIR = OUTPUT_DIR / "learned_networks"
LEARNING_OUTPUT_DIR = OUTPUT_DIR / "learning"
INFERENCE_OUTPUT_DIR = OUTPUT_DIR / "inference"

LEARNED_MODEL_PATH = LEARNED_NETWORK_DIR / "breast_cancer_learned.xml"
LEARNING_HISTORY_PATH = LEARNING_OUTPUT_DIR / "history.csv"
SELECTED_CPT_CSV_PATH = LEARNING_OUTPUT_DIR / "learned_fatigue_cpt.csv"
SELECTED_CPT_TEX_PATH = LEARNING_OUTPUT_DIR / "learned_fatigue_cpt.tex"
ADVERSE_INFERENCE_CSV_PATH = INFERENCE_OUTPUT_DIR / "treatment_comparison_adverse.csv"
ADVERSE_INFERENCE_TEX_PATH = INFERENCE_OUTPUT_DIR / "treatment_comparison_adverse.tex"
FAVORABLE_INFERENCE_CSV_PATH = INFERENCE_OUTPUT_DIR / "treatment_comparison_favorable.csv"
FAVORABLE_INFERENCE_TEX_PATH = INFERENCE_OUTPUT_DIR / "treatment_comparison_favorable.tex"

for directory in [LEARNED_NETWORK_DIR, LEARNING_OUTPUT_DIR, INFERENCE_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Learning configuration
# ---------------------------------------------------------------------
FIXED_CPD_NODES = [
    "Virtual Evidence Fatigue",
    "Virtual Evidence Pain",
    "Virtual Evidence Body Image",
]

DEFAULT_PRIOR_STRENGTH = 1.0
NODE_STRENGTH_OVERRIDES = {}

MAP_EM_MAX_ITER = 50
MAP_EM_TOL = 1e-6
SOFT_EPSILON = 1e-6

# ---------------------------------------------------------------------
# CPT excerpt
# ---------------------------------------------------------------------
SELECTED_CPT_NODE = "Fatigue Post"
SELECTED_CPT_PARENT_CONFIGURATION = {
    "Adjuvant therapy": "chemotherapy",
    "Fatigue Pre": "moderate",
    "Surgery": "mastectomy",
}

# ---------------------------------------------------------------------
# Inference configuration
# ---------------------------------------------------------------------
TREATMENT_VARS = [
    "Adjuvant therapy",
    "Surgery",
]

QOL_DOMAINS = OrderedDict({
    "fatigue": {
        "pre": "Fatigue Pre",
        "post": "Fatigue Post",
        "follow": "Fatigue Follow",
        "statement": "Virtual Evidence Fatigue",
        "worse_state": "increased",
        "not_worse_state": "decreased_or_similar",
    },
    "pain": {
        "pre": "Pain Pre",
        "post": "Pain Post",
        "follow": "Pain Follow",
        "statement": "Virtual Evidence Pain",
        "worse_state": "increased",
        "not_worse_state": "decreased_or_similar",
    },
    "body_image": {
        "pre": "Body Image Pre",
        "post": "Body Image Post",
        "follow": "Body Image Follow",
        "statement": "Virtual Evidence Body Image",
        "worse_state": "decreased",
        "not_worse_state": "increased_or_similar",
    },
})

TARGET_STATES_ADVERSE = {
    "fatigue": "increased",
    "pain": "increased",
    "body_image": "decreased",
}

TARGET_STATES_FAVORABLE = {
    "fatigue": "decreased_or_similar",
    "pain": "decreased_or_similar",
    "body_image": "increased_or_similar",
}


## 3. Load Bayesian network and SOAP-derived evidence


In [ ]:
def read_bayesian_network(path):
    """Load a discrete Bayesian network from XMLBIF or BIF format."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Model file not found: {path}")

    readers = [XMLBIFReader, BIFReader]
    last_error = None

    for reader_cls in readers:
        try:
            reader = reader_cls(str(path))
            return reader.get_model(state_name_type=str)
        except Exception as exc:
            last_error = exc

    raise ValueError(f"Could not read Bayesian network from {path}.") from last_error


def extract_metadata(model):
    """Extract state names, parent order, and CPT shapes from a pgmpy model."""
    metadata = {}

    for node in model.nodes():
        cpd = model.get_cpds(node)
        if cpd is None:
            raise ValueError(f"No CPD found for node: {node}")

        parents = list(cpd.variables[1:])
        node_states = list(cpd.state_names[node])
        parent_states = {p: list(cpd.state_names[p]) for p in parents}

        if parents:
            parent_combos = list(itertools.product(*[parent_states[p] for p in parents]))
        else:
            parent_combos = [()]

        metadata[node] = {
            "node": node,
            "parents": parents,
            "node_states": node_states,
            "parent_states": parent_states,
            "node_card": len(node_states),
            "parent_combos": parent_combos,
            "n_parent_configs": len(parent_combos),
            "shape": (len(node_states), len(parent_combos)),
            "node_state_to_idx": {s: i for i, s in enumerate(node_states)},
            "parent_combo_to_col": {combo: j for j, combo in enumerate(parent_combos)},
        }

    return metadata


def normalize_missing_tokens(df):
    missing_tokens = {"", "na", "n/a", "nan", "none", "null", "unknown"}
    df = df.copy()

    for col in df.columns:
        if col == "_weight":
            continue

        def _clean(x):
            if pd.isna(x):
                return np.nan
            if isinstance(x, str):
                x = x.strip()
                return np.nan if x.lower() in missing_tokens else x
            return x

        df[col] = df[col].map(_clean)

    return df


def load_evidence_table(path, model, metadata):
    """Load and validate the SOAP-derived evidence table."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Evidence file not found: {path}")

    data = pd.read_csv(path, sep=",", dtype=object)
    data = normalize_missing_tokens(data)

    if "_weight" in data.columns:
        data["_weight"] = pd.to_numeric(data["_weight"], errors="raise").fillna(1.0)
        if (data["_weight"] <= 0).any():
            raise ValueError("All _weight values must be > 0.")
    else:
        data["_weight"] = 1.0

    model_nodes = list(model.nodes())

    for node in model_nodes:
        if node not in data.columns:
            data[node] = np.nan

    unexpected_cols = sorted(set(data.columns) - set(model_nodes) - {"_weight"})
    if unexpected_cols:
        raise ValueError(f"Unexpected columns in evidence CSV: {unexpected_cols}")

    data = data[model_nodes + ["_weight"]]

    invalid_entries = []
    for node in model_nodes:
        allowed = set(metadata[node]["node_states"])
        observed_values = pd.Series(data[node].dropna().unique()).tolist()
        invalid = [v for v in observed_values if v not in allowed]
        if invalid:
            invalid_entries.append({
                "Variable": node,
                "InvalidValues": invalid,
                "AllowedStates": metadata[node]["node_states"],
            })

    if invalid_entries:
        invalid_df = pd.DataFrame(invalid_entries)
        display(invalid_df)
        raise ValueError("Evidence table contains invalid state labels.")

    return data


model = read_bayesian_network(MODEL_PATH)
assert model.check_model(), "The input Bayesian network failed check_model()."

metadata = extract_metadata(model)
data = load_evidence_table(EVIDENCE_PATH, model, metadata)

print(f"Loaded model: {len(model.nodes())} nodes, {len(model.edges())} edges")
print(f"Loaded evidence table: {data.shape[0]} rows, {data.shape[1]} columns")


## 4. Utility functions


In [ ]:
def cpd_to_frame(cpd):
    """Convert a TabularCPD into a pandas DataFrame."""
    values = cpd.get_values()
    node = cpd.variable
    node_states = cpd.state_names[node]
    parents = list(cpd.variables[1:])

    if parents:
        parent_combos = list(itertools.product(*[cpd.state_names[p] for p in parents]))
        columns = pd.MultiIndex.from_tuples(parent_combos, names=parents)
    else:
        columns = ["P"]

    return pd.DataFrame(values, index=node_states, columns=columns)


def write_latex_table(df, path, caption, label, column_format=None, escape=True):
    """Write a small manuscript table to LaTeX."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    latex = df.to_latex(
        index=False,
        escape=escape,
        caption=caption,
        label=label,
        column_format=column_format,
    )
    path.write_text(latex)
    return path


## 5. Informative Dirichlet prior


In [ ]:
def make_informative_dirichlet_alphas_from_model(
    model,
    metadata,
    default_prior_strength=10.0,
    node_strength_overrides=None,
    preserve_zero_probs=True,
    prior_floor=1e-12,
):
    """Create Dirichlet hyperparameters centered on the input CPTs."""
    node_strength_overrides = node_strength_overrides or {}
    alphas = {}

    for node, m in metadata.items():
        cpd = model.get_cpds(node)
        if cpd is None:
            raise ValueError(f"No CPD found for node: {node}")

        probs = np.asarray(cpd.get_values(), dtype=float)
        if probs.shape != m["shape"]:
            raise ValueError(
                f"Shape mismatch for node {node}: CPD has {probs.shape}, "
                f"metadata expects {m['shape']}"
            )

        col_sums = probs.sum(axis=0, keepdims=True)
        if np.any(col_sums <= 0):
            raise ValueError(f"Node {node} has a CPT column with non-positive total probability.")

        probs = probs / col_sums

        if not preserve_zero_probs:
            probs = np.maximum(probs, prior_floor)
            probs = probs / probs.sum(axis=0, keepdims=True)

        strength = float(node_strength_overrides.get(node, default_prior_strength))
        if strength <= 0:
            raise ValueError(f"Prior strength must be positive for node {node}.")

        alphas[node] = 1.0 + strength * probs

    return alphas


alphas = make_informative_dirichlet_alphas_from_model(
    model=model,
    metadata=metadata,
    default_prior_strength=DEFAULT_PRIOR_STRENGTH,
    node_strength_overrides=NODE_STRENGTH_OVERRIDES,
    preserve_zero_probs=True,
)


## 6. MAP-EM learning functions


In [ ]:
def expected_family_counts_for_row(
    infer,
    infer_soft,
    observed_evidence,
    node,
    m,
    row_index=None,
    fallback_hits=None,
):
    """
    E-step contribution for one node family on one patient row.

    Uses exact inference first.
    If the evidence has zero mass under the current model, falls back to a softly
    smoothed copy of the model to avoid a hard crash.

    Returns a matrix of shape: (node_cardinality, number_of_parent_configurations)
    """
    family_vars = [node] + m["parents"]

    family_observed = {v: observed_evidence[v] for v in family_vars if v in observed_evidence}
    hidden_family_vars = [v for v in family_vars if v not in family_observed]

    counts = np.zeros(m["shape"], dtype=float)

    # Fully observed family -> deterministic count in matching CPT cell
    if not hidden_family_vars:
        node_state = family_observed[node]
        parent_combo = tuple(family_observed[p] for p in m["parents"])
        counts[m["node_state_to_idx"][node_state], m["parent_combo_to_col"][parent_combo]] = 1.0
        return counts

    def _run_query(inference_engine):
        query_evidence = {k: v for k, v in observed_evidence.items() if k not in hidden_family_vars}
        factor = inference_engine.query(
            variables=hidden_family_vars,
            evidence=query_evidence,
            show_progress=False,
        )
        values = np.asarray(factor.values, dtype=float)
        total_mass = float(values.sum())
        return factor, values, total_mass

    used_soft_fallback = False

    # First try exact inference on current model
    try:
        factor, values, total_mass = _run_query(infer)
    except Exception:
        factor, values, total_mass = None, None, 0.0

    # If impossible under current model, retry on softened model
    if (values is None) or (not np.isfinite(total_mass)) or (total_mass <= 0):
        try:
            factor, values, total_mass = _run_query(infer_soft)
            used_soft_fallback = True
        except Exception:
            factor, values, total_mass = None, None, 0.0

    if (values is None) or (not np.isfinite(total_mass)) or (total_mass <= 0):
        raise ValueError(
            f"Row {row_index}: evidence remains impossible even after soft fallback for node {node}. "
            f"Observed evidence was: {observed_evidence}"
        )

    if used_soft_fallback and fallback_hits is not None:
        fallback_hits.append(
            {
                "row_index": row_index,
                "node": node,
                "evidence": dict(observed_evidence),
            }
        )

    for idx in np.ndindex(values.shape):
        prob = float(values[idx])
        if prob <= 0:
            continue

        assignment = dict(family_observed)
        for axis, var in enumerate(factor.variables):
            state_label = factor.state_names[var][idx[axis]]
            assignment[var] = state_label

        node_state = assignment[node]
        parent_combo = tuple(assignment[p] for p in m["parents"])

        row_idx = m["node_state_to_idx"][node_state]
        col_idx = m["parent_combo_to_col"][parent_combo]
        counts[row_idx, col_idx] += prob

    return counts


def map_update_from_counts(counts, alpha, min_prob=1e-12):
    """
    MAP mode update:
        theta = (count + alpha - 1) / normalization
    """
    numer = counts + alpha - 1.0
    numer = np.maximum(numer, min_prob)
    probs = numer / numer.sum(axis=0, keepdims=True)
    return probs


def apply_constraints_to_probs(node, probs, m):
    """
    Hard clinical constraints applied after the MAP update.
    """
    probs = probs.copy()

    if node == "Tumour Stage" and "Metastasis" in m["parents"] and "IV" in m["node_states"]:
        met_parent_pos = m["parents"].index("Metastasis")
        iv_row = m["node_state_to_idx"]["IV"]

        for col, combo in enumerate(m["parent_combos"]):
            metastasis_state = combo[met_parent_pos]

            if metastasis_state == "yes":
                probs[:, col] = 0.0
                probs[iv_row, col] = 1.0

            elif metastasis_state == "no":
                probs[iv_row, col] = 0.0
                col_sum = probs[:, col].sum()

                if col_sum <= 0:
                    probs[:, col] = 0.0
                    non_iv_rows = [r for r in range(len(m["node_states"])) if r != iv_row]
                    probs[non_iv_rows, col] = 1.0 / len(non_iv_rows)
                else:
                    probs[:, col] /= col_sum

    return probs


def make_cpd_from_probs(node, probs, m):
    kwargs = {
        "variable": node,
        "variable_card": m["node_card"],
        "values": probs,
        "state_names": {node: m["node_states"], **{p: m["parent_states"][p] for p in m["parents"]}},
    }

    if m["parents"]:
        kwargs["evidence"] = m["parents"]
        kwargs["evidence_card"] = [len(m["parent_states"][p]) for p in m["parents"]]

    return TabularCPD(**kwargs)


def rebuild_model_with_cpds(structure_model, cpds):
    new_model = DiscreteBayesianNetwork(structure_model.edges())
    new_model.add_nodes_from(structure_model.nodes())
    new_model.add_cpds(*cpds)
    assert new_model.check_model(), "New model failed check_model()."
    return new_model


def make_softened_model(structure_model, epsilon=1e-6, preserve_constraints=True):
    """
    Creates a softened copy of the current model by flooring every nonzero-free CPT cell
    at epsilon and renormalizing each column. This is used as a fallback during the
    E-step when a row has impossible evidence under the exact current model.
    """
    mdata = extract_metadata(structure_model)
    new_cpds = []

    for node in structure_model.nodes():
        old_cpd = structure_model.get_cpds(node)
        probs = np.asarray(old_cpd.get_values(), dtype=float).copy()

        probs = np.maximum(probs, epsilon)
        probs /= probs.sum(axis=0, keepdims=True)

        if preserve_constraints:
            probs = apply_constraints_to_probs(node, probs, mdata[node])

        new_cpds.append(make_cpd_from_probs(node, probs, mdata[node]))

    softened_model = rebuild_model_with_cpds(structure_model, new_cpds)
    return softened_model

def map_em(
    initial_model,
    data,
    alphas,
    max_iter=50,
    tol=1e-6,
    enforce_constraints=True,
    soft_epsilon=1e-6,
    fixed_cpd_nodes=None,
    verbose=True,
):
    """
    MAP-EM for a fully discrete BN with missing observations.

    Uses exact inference by default.
    Falls back to a softly-smoothed model for rows whose observed evidence has zero
    probability under the current exact model.

    fixed_cpd_nodes:
        Nodes whose CPDs must not be learned. This should include trajectory-level
        virtual evidence nodes, because their CPDs encode the verbal-statement
        semantics used later for inference.
    """
    current_model = initial_model
    metadata = extract_metadata(current_model)
    history = []
    fallback_log = []

    fixed_cpd_nodes = set(fixed_cpd_nodes or [])

    missing_fixed_nodes = [node for node in fixed_cpd_nodes if node not in current_model.nodes()]
    if missing_fixed_nodes:
        raise ValueError(f"fixed_cpd_nodes contains nodes not in the model: {missing_fixed_nodes}")

    for iteration in range(1, max_iter + 1):
        infer = VariableElimination(current_model)

        softened_model = make_softened_model(
            current_model,
            epsilon=soft_epsilon,
            preserve_constraints=enforce_constraints,
        )
        infer_soft = VariableElimination(softened_model)

        expected_counts = {
            node: np.zeros(metadata[node]["shape"], dtype=float)
            for node in current_model.nodes()
            if node not in fixed_cpd_nodes
        }

        iter_fallback_hits = []

        for row_index, row in data.iterrows():
            observed_evidence = {
                node: row[node]
                for node in current_model.nodes()
                if pd.notna(row[node])
            }
            row_weight = float(row["_weight"]) if "_weight" in row else 1.0

            for node in current_model.nodes():
                if node in fixed_cpd_nodes:
                    continue

                counts = expected_family_counts_for_row(
                    infer=infer,
                    infer_soft=infer_soft,
                    observed_evidence=observed_evidence,
                    node=node,
                    m=metadata[node],
                    row_index=row_index,
                    fallback_hits=iter_fallback_hits,
                )
                expected_counts[node] += row_weight * counts

        new_cpds = []
        max_delta = 0.0

        for node in current_model.nodes():
            m = metadata[node]
            old_probs = current_model.get_cpds(node).get_values()

            if node in fixed_cpd_nodes:
                probs = old_probs.copy()
            else:
                probs = map_update_from_counts(expected_counts[node], alphas[node])

                if enforce_constraints:
                    probs = apply_constraints_to_probs(node, probs, m)

                max_delta = max(max_delta, float(np.max(np.abs(probs - old_probs))))

            new_cpds.append(make_cpd_from_probs(node, probs, m))

        current_model = rebuild_model_with_cpds(current_model, new_cpds)

        history.append({
            "iteration": iteration,
            "max_cpd_delta": max_delta,
            "soft_fallback_count": len(iter_fallback_hits),
        })

        fallback_log.extend(
            [{"iteration": iteration, **item} for item in iter_fallback_hits]
        )

        if verbose:
            print(
                f"Iteration {iteration:03d} | "
                f"max CPD delta = {max_delta:.10f} | "
                f"soft fallback hits = {len(iter_fallback_hits)}"
            )

        if max_delta < tol:
            if verbose:
                print("Converged.")
            break

    history_df = pd.DataFrame(history)
    fallback_df = pd.DataFrame(fallback_log)
    return current_model, history_df, expected_counts, fallback_df


## 7. Run MAP-EM learning


In [ ]:
LEARNED_MODEL, HISTORY, EXPECTED_COUNTS, FALLBACK_LOG = map_em(
    initial_model=model,
    data=data,
    alphas=alphas,
    max_iter=MAP_EM_MAX_ITER,
    tol=MAP_EM_TOL,
    enforce_constraints=True,
    soft_epsilon=SOFT_EPSILON,
    fixed_cpd_nodes=FIXED_CPD_NODES,
    verbose=False,
)

assert LEARNED_MODEL.check_model(), "Learned model failed check_model()."

HISTORY.to_csv(LEARNING_HISTORY_PATH, index=False)
XMLBIFWriter(LEARNED_MODEL).write(str(LEARNED_MODEL_PATH))

print("MAP-EM completed")
print(f"Iterations: {len(HISTORY)}")
print(f"Final max CPT delta: {HISTORY['max_cpd_delta'].iloc[-1]:.6g}")
print(f"Saved learned network to: {LEARNED_MODEL_PATH}")
print(f"Saved learning history to: {LEARNING_HISTORY_PATH}")

if len(FALLBACK_LOG) > 0:
    FALLBACK_LOG.to_csv(LEARNING_OUTPUT_DIR / "soft_fallback_log.csv", index=False)
    print(f"Soft fallback cases logged: {len(FALLBACK_LOG)}")


## 8. MAP-EM convergence figure

In [ ]:
def plot_map_em_convergence(
    history,
    output_dir=LEARNING_OUTPUT_DIR,
    basename="map_em_convergence",
    tol=1e-6,
    dpi=600,
):
    """
    Plot the maximum absolute CPT change over MAP-EM iterations.
    """

    if history is None or len(history) == 0:
        raise ValueError("HISTORY is empty. Run MAP-EM before plotting convergence.")

    required_columns = {"iteration", "max_cpd_delta"}
    missing = required_columns - set(history.columns)
    if missing:
        raise ValueError(f"HISTORY is missing required columns: {missing}")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    plot_df = history.copy()
    plot_df["iteration"] = pd.to_numeric(plot_df["iteration"])
    plot_df["max_cpd_delta"] = pd.to_numeric(plot_df["max_cpd_delta"])

    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    fig, ax = plt.subplots(figsize=(4.2, 3.0), constrained_layout=True)

    ax.plot(
        plot_df["iteration"],
        plot_df["max_cpd_delta"],
        marker="o",
        linewidth=1.5,
        markersize=3.5,
    )

    ax.axhline(
        tol,
        linestyle="--",
        linewidth=1.0,
        label=f"Tolerance = {tol:g}",
    )

    ax.set_yscale("log")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Maximum absolute CPT change")
    ax.set_title("MAP-EM convergence")
    ax.grid(True, which="both", linewidth=0.4, alpha=0.5)
    ax.legend(frameon=False)

    png_path = output_dir / f"{basename}.png"
    pdf_path = output_dir / f"{basename}.pdf"

    fig.savefig(png_path, bbox_inches="tight", dpi=dpi)
    fig.savefig(pdf_path, bbox_inches="tight", dpi=dpi)
    plt.show()

    return {
        "png": png_path,
        "pdf": pdf_path,
    }


CONVERGENCE_FIGURE_PATHS = plot_map_em_convergence(HISTORY)

print("Saved MAP-EM convergence figure:")
for path in CONVERGENCE_FIGURE_PATHS.values():
    print(f"- {path}")

## 9. Learning output


In [ ]:
def extract_cpt_row(model, node, parent_configuration):
    """Extract one CPT row as a DataFrame with state probabilities."""
    m = extract_metadata(model)[node]
    cpd = model.get_cpds(node)
    probs = np.asarray(cpd.get_values(), dtype=float)

    parents = m["parents"]
    missing = [p for p in parents if p not in parent_configuration]
    extra = [p for p in parent_configuration if p not in parents]

    if missing:
        raise ValueError(f"Parent configuration missing parents for {node}: {missing}")
    if extra:
        raise ValueError(f"Parent configuration contains non-parents for {node}: {extra}")

    parent_combo = tuple(parent_configuration[p] for p in parents)
    if parent_combo not in m["parent_combo_to_col"]:
        raise ValueError(f"Invalid parent configuration for {node}: {parent_configuration}")

    col_idx = m["parent_combo_to_col"][parent_combo]

    return pd.DataFrame({
        "State of Fatigue Post": m["node_states"],
        "probability": probs[:, col_idx],
    })


def make_selected_cpt_comparison(initial_model, learned_model, node, parent_configuration):
    initial = extract_cpt_row(initial_model, node, parent_configuration)
    learned = extract_cpt_row(learned_model, node, parent_configuration)

    out = pd.DataFrame({
        "State of Fatigue Post": initial["State of Fatigue Post"],
        "Initial probability": initial["probability"],
        "Learned probability": learned["probability"],
    })

    return out


selected_cpt_table = make_selected_cpt_comparison(
    initial_model=model,
    learned_model=LEARNED_MODEL,
    node=SELECTED_CPT_NODE,
    parent_configuration=SELECTED_CPT_PARENT_CONFIGURATION,
)

selected_cpt_table.to_csv(SELECTED_CPT_CSV_PATH, index=False)
write_latex_table(
    selected_cpt_table,
    SELECTED_CPT_TEX_PATH,
    caption=(
        "Selected initial and learned probabilities for a fatigue-related CPT after learning "
        "from SOAP-derived virtual evidence."
    ),
    label="tab:learned_fatigue_cpt",
    column_format="lll",
    escape=True,
)

display(selected_cpt_table)
print(f"Saved selected CPT table to: {SELECTED_CPT_CSV_PATH}")
print(f"Saved selected CPT LaTeX table to: {SELECTED_CPT_TEX_PATH}")
print(f"Saved convergence graph PNG to: {CONVERGENCE_FIGURE_PATHS['png']}")
print(f"Saved convergence graph PDF to: {CONVERGENCE_FIGURE_PATHS['pdf']}")


# Inference

## 10. Exact inference utilities


In [ ]:
MODEL_INF = LEARNED_MODEL


def model_nodes(model=MODEL_INF):
    return list(model.nodes())


def get_cpd(model, node):
    cpd = model.get_cpds(node)
    if cpd is None:
        raise ValueError(f"No CPD found for node: {node}")
    return cpd


def node_states(node, model=MODEL_INF):
    cpd = get_cpd(model, node)
    if hasattr(cpd, "state_names") and node in cpd.state_names:
        return list(cpd.state_names[node])
    return list(range(cpd.variable_card))


def node_parents(node, model=MODEL_INF):
    cpd = get_cpd(model, node)
    if hasattr(cpd, "variables"):
        return list(cpd.variables[1:])
    if hasattr(cpd, "get_evidence"):
        return list(cpd.get_evidence())
    return list(model.get_parents(node))


def validate_node_exists(node, model=MODEL_INF):
    if node not in model.nodes():
        raise KeyError(f"Node `{node}` is not in the inference model.")


def validate_node_value(node, value, model=MODEL_INF):
    validate_node_exists(node, model=model)
    states = node_states(node, model=model)

    if value not in states:
        raise ValueError(
            f"Invalid state `{value}` for node `{node}`. "
            f"Allowed states: {states}"
        )


def validate_domain_config(model=MODEL_INF):
    errors = []

    for treatment_var in TREATMENT_VARS:
        if treatment_var not in model.nodes():
            errors.append(f"Treatment variable missing: {treatment_var}")

    for domain, cfg in QOL_DOMAINS.items():
        required = [cfg["pre"], cfg["post"], cfg["follow"], cfg["statement"]]
        for node in required:
            if node not in model.nodes():
                errors.append(f"[{domain}] missing node: {node}")

        if cfg["statement"] in model.nodes():
            expected_parents = [cfg["pre"], cfg["post"], cfg["follow"]]
            actual_parents = node_parents(cfg["statement"], model=model)
            missing_parents = [p for p in expected_parents if p not in actual_parents]
            extra_parents = [p for p in actual_parents if p not in expected_parents]

            if missing_parents:
                errors.append(f"[{domain}] statement node missing parents: {missing_parents}")
            if extra_parents:
                errors.append(f"[{domain}] statement node has unexpected parents: {extra_parents}")

            states = node_states(cfg["statement"], model=model)
            for key in ["worse_state", "not_worse_state"]:
                if cfg[key] not in states:
                    errors.append(f"[{domain}] {key} `{cfg[key]}` not in states of `{cfg['statement']}`: {states}")

    if errors:
        raise ValueError("\n".join(errors))


validate_domain_config(MODEL_INF)
print("Inference configuration validated.")


In [ ]:
def domain_config(domain):
    if domain not in QOL_DOMAINS:
        raise KeyError(f"Unknown domain `{domain}`. Available domains: {list(QOL_DOMAINS.keys())}")
    return QOL_DOMAINS[domain]


def resolve_domain_statement_node(domain):
    return domain_config(domain)["statement"]


def resolve_domain_statement_state(domain, target="worse"):
    cfg = domain_config(domain)

    if target in ["worse", "adverse", "active"]:
        return cfg["worse_state"]

    if target in ["not_worse", "not adverse", "inactive", "favourable", "favorable"]:
        return cfg["not_worse_state"]

    # Direct state supplied.
    statement_node = cfg["statement"]
    validate_node_value(statement_node, target)
    return target


def normalize_evidence_dict(evidence, validate=True):
    if evidence is None:
        return {}

    normalized = {}

    for node, value in evidence.items():
        if value is None or pd.isna(value):
            continue

        if validate:
            validate_node_value(node, value)

        normalized[node] = value

    return normalized


def normalize_domain_statement_evidence(statement_evidence, validate=True):
    """
    Accepts either exact statement-node evidence:

        {"Virtual Evidence Fatigue": "increased"}

    or domain-level evidence:

        {"fatigue": "worse"}
        {"pain": "not_worse"}
        {"body_image": "worse"}

    Returns ordinary BN evidence.
    """
    if statement_evidence is None:
        return {}

    normalized = {}

    for key, value in statement_evidence.items():
        if value is None or pd.isna(value):
            continue

        if key in QOL_DOMAINS:
            statement_node = resolve_domain_statement_node(key)
            statement_state = resolve_domain_statement_state(key, target=value)
        else:
            statement_node = key
            statement_state = value

        if validate:
            validate_node_value(statement_node, statement_state)

        normalized[statement_node] = statement_state

    return normalized


def normalize_treatment_do(treatment_do, validate=True):
    """
    Normalizes treatment intervention dictionaries.

    Example:
        {
            "Adjuvant therapy": "chemotherapy",
            "Surgery": "breast_conserving",
        }
    """
    if treatment_do is None:
        return {}

    normalized = {}

    for node, value in treatment_do.items():
        if node not in TREATMENT_VARS:
            raise KeyError(
                f"`{node}` is not configured as a treatment variable. "
                f"Allowed treatment variables: {TREATMENT_VARS}"
            )

        if value is None or pd.isna(value):
            continue

        if validate:
            validate_node_value(node, value)

        normalized[node] = value

    return normalized


def build_scenario(
    clinical_evidence=None,
    statement_evidence=None,
    treatment_do=None,
    treatment_as_evidence=None,
    validate=True,
):
    """
    Builds a reusable inference/simulation scenario.

    clinical_evidence:
        Ordinary observed patient variables.

    statement_evidence:
        Evidence on trajectory-level virtual evidence nodes, either by exact node
        name or by domain shorthand.

    treatment_do:
        Interventional treatment assignment, e.g.
        {"Adjuvant therapy": "chemotherapy", "Surgery": "breast_conserving"}.

    treatment_as_evidence:
        Observational treatment evidence. Use this only for conditioning, not
        causal/interventional queries.

    Returns:
        {
            "evidence": dict,
            "do": dict,
            "metadata": dict
        }
    """
    evidence = {}

    evidence.update(normalize_evidence_dict(clinical_evidence, validate=validate))
    evidence.update(normalize_domain_statement_evidence(statement_evidence, validate=validate))
    evidence.update(normalize_evidence_dict(treatment_as_evidence, validate=validate))

    do = normalize_treatment_do(treatment_do, validate=validate)

    # A variable cannot simultaneously be evidence and intervention.
    for intervened_node in do:
        evidence.pop(intervened_node, None)

    return {
        "evidence": evidence,
        "do": do,
        "metadata": {
            "clinical_evidence": clinical_evidence or {},
            "statement_evidence": statement_evidence or {},
            "treatment_do": treatment_do or {},
            "treatment_as_evidence": treatment_as_evidence or {},
        },
    }


def scenario_from_row(
    row,
    treatment_do=None,
    include_statement_nodes=True,
    validate=True,
):
    """
    Converts a dataframe row into scenario evidence.

    Observed model variables are used as evidence.
    If treatment_do is supplied, those treatment variables are removed from
    evidence and placed in the intervention dictionary.
    """
    evidence = {}

    excluded = set(treatment_do.keys()) if treatment_do else set()

    for node in MODEL_INF.nodes():
        if node not in row.index:
            continue

        if node in excluded:
            continue

        if not include_statement_nodes and node in STATEMENT_NODES:
            continue

        value = row[node]

        if pd.isna(value):
            continue

        if validate:
            validate_node_value(node, value)

        evidence[node] = value

    return build_scenario(
        clinical_evidence=evidence,
        statement_evidence=None,
        treatment_do=treatment_do,
        treatment_as_evidence=None,
        validate=validate,
    )


In [ ]:
INFER = VariableElimination(MODEL_INF)


def scenario_evidence(scenario_or_evidence=None):
    if scenario_or_evidence is None:
        return {}

    if isinstance(scenario_or_evidence, dict) and "evidence" in scenario_or_evidence:
        return scenario_or_evidence["evidence"]

    return normalize_evidence_dict(scenario_or_evidence, validate=True)


def query_factor(variables, scenario_or_evidence=None):
    evidence = scenario_evidence(scenario_or_evidence)

    if isinstance(variables, str):
        variables = [variables]

    for var in variables:
        validate_node_exists(var)

    return INFER.query(
        variables=list(variables),
        evidence=evidence,
        show_progress=False,
    )


def query_marginal(variable, scenario_or_evidence=None):
    return query_factor([variable], scenario_or_evidence=scenario_or_evidence)


def query_map(variables, scenario_or_evidence=None):
    evidence = scenario_evidence(scenario_or_evidence)

    if isinstance(variables, str):
        variables = [variables]

    for var in variables:
        validate_node_exists(var)

    return INFER.map_query(
        variables=list(variables),
        evidence=evidence,
        show_progress=False,
    )


def factor_to_dataframe(factor):
    variables = list(factor.variables)
    state_names = factor.state_names

    rows = []

    for idx in np.ndindex(factor.values.shape):
        row = {
            var: state_names[var][idx[axis]]
            for axis, var in enumerate(variables)
        }
        row["probability"] = float(factor.values[idx])
        rows.append(row)

    return pd.DataFrame(rows)


def query_dataframe(variables, scenario_or_evidence=None, sort=True):
    factor = query_factor(variables, scenario_or_evidence=scenario_or_evidence)
    df = factor_to_dataframe(factor)

    if sort:
        df = df.sort_values("probability", ascending=False).reset_index(drop=True)

    return df


def query_domain_statement_probability(
    domain,
    scenario_or_evidence=None,
    target="worse",
):
    """
    Computes:

        P(statement_domain = target_state | evidence)

    Example:
        query_domain_statement_probability("fatigue", target="worse")
        query_domain_statement_probability("pain", target="not_worse")
        query_domain_statement_probability("body_image", target="worse")
    """
    cfg = domain_config(domain)
    statement_node = cfg["statement"]
    target_state = resolve_domain_statement_state(domain, target=target)

    factor = query_marginal(statement_node, scenario_or_evidence=scenario_or_evidence)
    states = factor.state_names[statement_node]

    idx = states.index(target_state)
    return float(factor.values[idx])


def query_all_domain_statement_probabilities(
    scenario_or_evidence=None,
    target="worse",
):
    rows = []

    for domain, cfg in QOL_DOMAINS.items():
        statement_node = cfg["statement"]
        target_state = resolve_domain_statement_state(domain, target=target)

        rows.append({
            "domain": domain,
            "statement_node": statement_node,
            "target_state": target_state,
            "probability": query_domain_statement_probability(
                domain,
                scenario_or_evidence=scenario_or_evidence,
                target=target,
            ),
        })

    return pd.DataFrame(rows).sort_values("probability", ascending=False).reset_index(drop=True)


def query_domain_trajectory(domain, scenario_or_evidence=None):
    cfg = domain_config(domain)

    variables = [
        cfg["pre"],
        cfg["post"],
        cfg["follow"],
    ]

    return query_factor(variables, scenario_or_evidence=scenario_or_evidence)


def query_domain_trajectory_dataframe(domain, scenario_or_evidence=None):
    return factor_to_dataframe(query_domain_trajectory(domain, scenario_or_evidence)).sort_values(
        "probability",
        ascending=False,
    ).reset_index(drop=True)


In [ ]:
def treatment_scenario_label(treatment_do):
    if not treatment_do:
        return "no_treatment_setting"

    return " | ".join(
        f"{node}={value}"
        for node, value in treatment_do.items()
    )


def statement_node_states(statement_node):
    cpd = MODEL_INF.get_cpds(statement_node)
    if cpd is None:
        raise ValueError(f"No CPD found for statement node `{statement_node}`.")

    return list(cpd.state_names[statement_node])


def validate_statement_target(statement_node, target_state):
    states = statement_node_states(statement_node)

    if target_state not in states:
        raise ValueError(
            f"Invalid target state `{target_state}` for `{statement_node}`. "
            f"Allowed states: {states}"
        )

    return target_state


def make_intervened_model(model, treatment_do):
    """
    Create an intervened copy of the Bayesian network for a treatment setting.

    Incoming edges into treatment variables are removed. Each treatment variable
    is assigned a deterministic root CPD with probability 1 for the chosen
    treatment value. Outgoing edges are preserved.
    """
    treatment_do = normalize_treatment_do(treatment_do, validate=True)
    intervened_nodes = set(treatment_do.keys())

    new_edges = [
        (u, v)
        for (u, v) in model.edges()
        if v not in intervened_nodes
    ]

    intervened_model = DiscreteBayesianNetwork(new_edges)
    intervened_model.add_nodes_from(model.nodes())

    new_cpds = []

    for node in model.nodes():
        old_cpd = model.get_cpds(node)

        if node in intervened_nodes:
            states = list(old_cpd.state_names[node])
            fixed_value = treatment_do[node]

            if fixed_value not in states:
                raise ValueError(
                    f"Invalid treatment value `{fixed_value}` for `{node}`. "
                    f"Allowed states: {states}"
                )

            values = [
                [1.0 if state == fixed_value else 0.0]
                for state in states
            ]

            new_cpd = TabularCPD(
                variable=node,
                variable_card=len(states),
                values=values,
                state_names={node: states},
            )

        else:
            new_cpd = copy.deepcopy(old_cpd)

        new_cpds.append(new_cpd)

    intervened_model.add_cpds(*new_cpds)

    if not intervened_model.check_model():
        raise ValueError("Intervened model failed check_model().")

    return intervened_model


def estimate_domain_statement_under_treatment(
    domain,
    treatment_do,
    target_state,
    clinical_evidence=None,
    statement_evidence=None,
):
    """
    Computes exactly:

        Pr(statement_node = target_state | clinical evidence, treatment setting)

    `target_state` must be the literal state of the virtual evidence node.

    Examples:
        domain="fatigue",    target_state="increased"
        domain="pain",       target_state="decreased_or_similar"
        domain="body_image", target_state="increased_or_similar"
    """
    cfg = domain_config(domain)
    statement_node = cfg["statement"]
    target_state = validate_statement_target(statement_node, target_state)

    scenario = build_scenario(
        clinical_evidence=clinical_evidence,
        statement_evidence=statement_evidence,
        treatment_do=treatment_do,
        treatment_as_evidence=None,
        validate=True,
    )

    treatment_do = scenario["do"]
    evidence = dict(scenario["evidence"])

    # Treatment variables are fixed in the intervened model, not conditioned on.
    for node in treatment_do:
        evidence.pop(node, None)

    # If the queried statement node is already evidence, the query is degenerate.
    if statement_node in evidence:
        observed_state = evidence[statement_node]
        probability = 1.0 if observed_state == target_state else 0.0

        return {
            "domain": domain,
            "statement_node": statement_node,
            "target_state": target_state,
            "treatment_scenario": treatment_scenario_label(treatment_do),
            "treatment_do": treatment_do,
            "probability": float(probability),
            "method": "exact_variable_elimination_degenerate",
            "evidence": evidence,
        }

    intervened_model = make_intervened_model(
        model=MODEL_INF,
        treatment_do=treatment_do,
    )

    infer = VariableElimination(intervened_model)

    factor = infer.query(
        variables=[statement_node],
        evidence=evidence,
        show_progress=False,
    )

    states = list(factor.state_names[statement_node])
    values = np.asarray(factor.values, dtype=float).reshape(-1)

    probability = float(values[states.index(target_state)])

    return {
        "domain": domain,
        "statement_node": statement_node,
        "target_state": target_state,
        "treatment_scenario": treatment_scenario_label(treatment_do),
        "treatment_do": treatment_do,
        "probability": probability,
        "method": "exact_variable_elimination",
        "evidence": evidence,
    }


def estimate_all_domain_statements_under_treatment(
    treatment_do,
    target_states,
    clinical_evidence=None,
    statement_evidence=None,
):
    """
    Computes exact statement probabilities for all QoL domains.

    `target_states` must specify the literal target state per domain.

    Example:
        target_states = {
            "fatigue": "increased",
            "pain": "increased",
            "body_image": "decreased",
        }
    """
    rows = []

    for domain in QOL_DOMAINS:
        if domain not in target_states:
            raise ValueError(
                f"No target state specified for domain `{domain}`. "
                f"Required domains: {list(QOL_DOMAINS.keys())}"
            )

        result = estimate_domain_statement_under_treatment(
            domain=domain,
            treatment_do=treatment_do,
            target_state=target_states[domain],
            clinical_evidence=clinical_evidence,
            statement_evidence=statement_evidence,
        )
        rows.append(result)

    return pd.DataFrame(rows)


In [ ]:
def all_treatment_scenarios():
    treatment_state_lists = [
        node_states(treatment_var)
        for treatment_var in TREATMENT_VARS
    ]

    scenarios = []

    for combo in itertools.product(*treatment_state_lists):
        scenarios.append({
            treatment_var: combo[i]
            for i, treatment_var in enumerate(TREATMENT_VARS)
        })

    return scenarios


def compare_treatment_scenarios_for_domain(
    domain,
    target_state,
    treatment_scenarios=None,
    clinical_evidence=None,
    statement_evidence=None,
    sort_ascending=True,
):
    """
    Computes exact probabilities for one literal virtual-evidence target state
    under all treatment settings.

    `target_state` must be a valid state of the virtual evidence node for
    the selected domain.
    """
    if treatment_scenarios is None:
        treatment_scenarios = all_treatment_scenarios()

    cfg = domain_config(domain)
    statement_node = cfg["statement"]
    target_state = validate_statement_target(statement_node, target_state)

    rows = []

    for treatment_do in treatment_scenarios:
        result = estimate_domain_statement_under_treatment(
            domain=domain,
            treatment_do=treatment_do,
            target_state=target_state,
            clinical_evidence=clinical_evidence,
            statement_evidence=statement_evidence,
        )

        row = {
            "domain": result["domain"],
            "statement_node": result["statement_node"],
            "target_state": result["target_state"],
            "treatment_scenario": result["treatment_scenario"],
            "probability": result["probability"],
            "method": result["method"],
        }

        row.update(result["treatment_do"])
        rows.append(row)

    out = pd.DataFrame(rows)
    out = out.sort_values("probability", ascending=sort_ascending).reset_index(drop=True)

    return out


def compare_treatment_scenarios_for_all_domains(
    target_states,
    treatment_scenarios=None,
    clinical_evidence=None,
    statement_evidence=None,
    sort_ascending=True,
):
    """
    Computes exact probabilities under all treatment settings for explicitly
    specified virtual-evidence target states.

    `target_states` must specify the literal target state per domain.

    Example:
        target_states = {
            "fatigue": "increased",
            "pain": "increased",
            "body_image": "decreased",
        }
    """
    all_rows = []

    for domain in QOL_DOMAINS:
        if domain not in target_states:
            raise ValueError(
                f"No target state specified for domain `{domain}`. "
                f"Required domains: {list(QOL_DOMAINS.keys())}"
            )

        df = compare_treatment_scenarios_for_domain(
            domain=domain,
            target_state=target_states[domain],
            treatment_scenarios=treatment_scenarios,
            clinical_evidence=clinical_evidence,
            statement_evidence=statement_evidence,
            sort_ascending=sort_ascending,
        )
        all_rows.append(df)

    return pd.concat(all_rows, ignore_index=True)


## 11. Inference output


In [ ]:
comparison_adverse = compare_treatment_scenarios_for_all_domains(
    target_states=TARGET_STATES_ADVERSE,
    clinical_evidence={},
    statement_evidence=None,
    sort_ascending=True,
)

comparison_favorable = compare_treatment_scenarios_for_all_domains(
    target_states=TARGET_STATES_FAVORABLE,
    clinical_evidence={},
    statement_evidence=None,
    sort_ascending=False,
)

comparison_adverse.to_csv(ADVERSE_INFERENCE_CSV_PATH, index=False)
comparison_favorable.to_csv(FAVORABLE_INFERENCE_CSV_PATH, index=False)

write_latex_table(
    comparison_adverse,
    ADVERSE_INFERENCE_TEX_PATH,
    caption="Exact treatment-setting inference for adverse temporal QoL statements.",
    label="tab:inference_adverse",
    escape=True,
)

write_latex_table(
    comparison_favorable,
    FAVORABLE_INFERENCE_TEX_PATH,
    caption="Exact treatment-setting inference for favourable temporal QoL statements.",
    label="tab:inference_favorable",
    escape=True,
)

print(f"Saved adverse inference table to: {ADVERSE_INFERENCE_CSV_PATH}")
print(f"Saved favorable inference table to: {FAVORABLE_INFERENCE_CSV_PATH}")

display(comparison_adverse)
display(comparison_favorable)


## 12. Reproducibility information and saved files


In [ ]:
def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"

reproducibility_info = pd.DataFrame([
    {"item": "run_timestamp", "value": datetime.now().isoformat(timespec="seconds")},
    {"item": "python", "value": platform.python_version()},
    {"item": "numpy", "value": package_version("numpy")},
    {"item": "pandas", "value": package_version("pandas")},
    {"item": "pgmpy", "value": package_version("pgmpy")},
    {"item": "model_path", "value": str(MODEL_PATH)},
    {"item": "evidence_path", "value": str(EVIDENCE_PATH)},
    {"item": "learned_model_path", "value": str(LEARNED_MODEL_PATH)},
])

reproducibility_info.to_csv(OUTPUT_DIR / "reproducibility_info.csv", index=False)
display(reproducibility_info)

print("Saved files:")
for path in [
    LEARNED_MODEL_PATH,
    LEARNING_HISTORY_PATH,
    SELECTED_CPT_CSV_PATH,
    SELECTED_CPT_TEX_PATH,
    ADVERSE_INFERENCE_CSV_PATH,
    ADVERSE_INFERENCE_TEX_PATH,
    FAVORABLE_INFERENCE_CSV_PATH,
    FAVORABLE_INFERENCE_TEX_PATH,
    OUTPUT_DIR / "reproducibility_info.csv",
]:
    print(f"- {path}")
